
- What types of files does the Dataset folder contain?
- How do I use them?
- What are some baseline models that I can implement?

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

print("✓ imports ready")

✓ imports ready


In [ ]:
# Set paths
base_path = Path("Dataset/Dataset/raw/elliptic_bitcoin_dataset")
features_file = base_path / "elliptic_txs_features.csv"
classes_file = base_path / "elliptic_txs_classes.csv"
edgelist_file = base_path / "elliptic_txs_edgelist.csv"

print(f"Loading features from {features_file}")
print(f"Loading classes from {classes_file}")

# Load features (167 columns: txId + 166 feature columns)
df_features = pd.read_csv(features_file, header=None)
print(f"Features shape: {df_features.shape}")

# Load classes - first row might be header
df_classes_raw = pd.read_csv(classes_file, header=None)
# Remove header row if it's 'txId', 'class'
if df_classes_raw.iloc[0, 0] == "txId":
    df_classes = df_classes_raw.iloc[1:].reset_index(drop=True)
else:
    df_classes = df_classes_raw

df_classes.columns = ["txId", "class"]
print(f"Classes shape: {df_classes.shape}")
print(f"Class distribution:\n{df_classes['class'].value_counts()}")



# Merge features with labels
# Column 0 is txId, columns 1-166 are features
df_features.columns = ["txId"] + [f"feature_{i}" for i in range(1, 167)]

# Ensure txId is the same type in both dataframes (convert to int)
df_features["txId"] = df_features["txId"].astype(int)
df_classes["txId"] = df_classes["txId"].astype(int)

df_data = df_features.merge(df_classes, on="txId", how="inner")

print(f"\nMerged data shape: {df_data.shape}")
print(f"Merged class distribution:\n{df_data['class'].value_counts()}")

Loading features from Dataset\Dataset\raw\elliptic_bitcoin_dataset\elliptic_txs_features.csv
Loading classes from Dataset\Dataset\raw\elliptic_bitcoin_dataset\elliptic_txs_classes.csv
Features shape: (203769, 167)
Classes shape: (203769, 2)
Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Merged data shape: (203769, 168)
Merged class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [ ]:
# Extract time step from feature_1 (temporal indicator)
df_data["time_step"] = df_data["feature_1"].astype(int)

# Keep only pre-shutdown period (timesteps 1-42); exclude timestep 43+
df_data = df_data[(df_data["time_step"] >= 1) & (df_data["time_step"] <= 42)].copy()

# Convert class to integer and filter out 'unknown' labels
# From the data: class 1 (4,545 samples) = illicit (minority), class 2 (42,019 samples) = licit (majority)
df_data["class_int"] = pd.to_numeric(df_data["class"], errors="coerce")
df_data = df_data[df_data["class_int"].notna()].copy()

print(f"After filtering to timesteps 1-42 and removing unknown labels: {df_data.shape[0]} samples")
print(f"  Class 1 (illicit): {(df_data['class_int'] == 1).sum()}")
print(f"  Class 2 (licit): {(df_data['class_int'] == 2).sum()}")
print(f"Time steps range used: {df_data['time_step'].min()} to {df_data['time_step'].max()}")

# Define time-forward splits: 32-5-5 over timesteps 1-42
train_times = (1, 32)
val_times = (33, 37)
test_times = (38, 42)

# Split by time step
mask_train = (df_data["time_step"] >= train_times[0]) & (
    df_data["time_step"] <= train_times[1]
)
mask_val = (df_data["time_step"] >= val_times[0]) & (
    df_data["time_step"] <= val_times[1]
)
mask_test = (df_data["time_step"] >= test_times[0]) & (
    df_data["time_step"] <= test_times[1]
)

df_train = df_data[mask_train].copy()
df_val = df_data[mask_val].copy()
df_test = df_data[mask_test].copy()

print(f"\n{'=' * 60}")
print("Time-Forward Split Results (32-5-5, pre-shutdown only):")
print(f"{'=' * 60}")
print(f"Train (t={train_times[0]}-{train_times[1]}):\t{df_train.shape[0]} samples")
print(
    f"  Class distribution: Illicit (1): {(df_train['class_int'] == 1).sum()}, Licit (2): {(df_train['class_int'] == 2).sum()}"
)

print(f"Val (t={val_times[0]}-{val_times[1]}):\t\t{df_val.shape[0]} samples")
print(
    f"  Class distribution: Illicit (1): {(df_val['class_int'] == 1).sum()}, Licit (2): {(df_val['class_int'] == 2).sum()}"
)

print(f"Test (t={test_times[0]}-{test_times[1]}):\t\t{df_test.shape[0]} samples")
print(
    f"  Class distribution: Illicit (1): {(df_test['class_int'] == 1).sum()}, Licit (2): {(df_test['class_int'] == 2).sum()}"
)

# Calculate illicit prevalence per split (class 1 is illicit)
illicit_counts = {
    "train": (df_train["class_int"] == 1).sum(),
    "val": (df_val["class_int"] == 1).sum(),
    "test": (df_test["class_int"] == 1).sum(),
}
total_counts = {"train": len(df_train), "val": len(df_val), "test": len(df_test)}
illicit_pct = {k: v / total_counts[k] * 100 for k, v in illicit_counts.items()}

print(f"\nIllicit (class=1) prevalence:")
print(f"  Train: {illicit_pct['train']:.2f}% ({illicit_counts['train']} samples)")
print(f"  Val:   {illicit_pct['val']:.2f}% ({illicit_counts['val']} samples)")
print(f"  Test:  {illicit_pct['test']:.2f}% ({illicit_counts['test']} samples)")
print(f"  → Prevalence drift: {illicit_pct['train']:.2f}% → {illicit_pct['test']:.2f}%")

After filtering unknown labels: 46564 samples
  Class 1 (illicit): 4545
  Class 2 (licit): 42019
Time steps range: 1 to 49

Time-Forward Split Results:
Train (t=1-34):	29894 samples
  Class distribution: Illicit (1): 3462, Licit (2): 26432
Val (t=35-41):		7829 samples
  Class distribution: Illicit (1): 675, Licit (2): 7154
Test (t=42-49):		8841 samples
  Class distribution: Illicit (1): 408, Licit (2): 8433

Illicit (class=1) prevalence:
  Train: 11.58% (3462 samples)
  Val:   8.62% (675 samples)
  Test:  4.61% (408 samples)
  → Prevalence drift: 11.58% → 4.61%


In [ ]:
# Extract feature matrix X and labels y
# Features are columns 2-166 (indices 1-165), excluding txId and feature_1 (time step)
# Class 1 = illicit, Class 2 = licit
feature_cols = [f"feature_{i}" for i in range(2, 167)]

X_train = df_train[feature_cols].values.astype(np.float32)
y_train = (df_train["class_int"] == 1).astype(int).values  # 1 = illicit, 0 = licit

X_val = df_val[feature_cols].values.astype(np.float32)
y_val = (df_val["class_int"] == 1).astype(int).values

X_test = df_test[feature_cols].values.astype(np.float32)
y_test = (df_test["class_int"] == 1).astype(int).values

print(f"Feature matrix shapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

print(f"\nLabel distribution (1=illicit, 0=licit):")
print(
    f"  Train: {(y_train == 1).sum()} illicit, {(y_train == 0).sum()} licit ({(y_train == 1).sum() / len(y_train) * 100:.2f}% illicit)"
)
print(
    f"  Val: {(y_val == 1).sum()} illicit, {(y_val == 0).sum()} licit ({(y_val == 1).sum() / len(y_val) * 100:.2f}% illicit)"
)
print(
    f"  Test: {(y_test == 1).sum()} illicit, {(y_test == 0).sum()} licit ({(y_test == 1).sum() / len(y_test) * 100:.2f}% illicit)"
)

# Compute class weights on training data only
classes = np.array([0, 1])
class_weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print(f"\nClass weights (computed from training data):")
print(f"  Class 0 (licit):  {class_weight_dict[0]:.4f}")
print(f"  Class 1 (illicit): {class_weight_dict[1]:.4f}")
print(
    f"  Weight ratio illicit/licit: {class_weight_dict[1] / class_weight_dict[0]:.2f}x"
)

Feature matrix shapes:
  X_train: (29894, 165), y_train: (29894,)
  X_val: (7829, 165), y_val: (7829,)
  X_test: (8841, 165), y_test: (8841,)

Label distribution (1=illicit, 0=licit):
  Train: 3462 illicit, 26432 licit (11.58% illicit)
  Val: 675 illicit, 7154 licit (8.62% illicit)
  Test: 408 illicit, 8433 licit (4.61% illicit)

Class weights (computed from training data):
  Class 0 (licit):  0.5655
  Class 1 (illicit): 4.3174
  Weight ratio illicit/licit: 7.63x


In [ ]:
X_train

In [5]:
# Feature standardization: fit on train, apply to all splits
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Feature standardization complete:")
print(f"  X_train mean: {X_train_scaled.mean():.6f}, std: {X_train_scaled.std():.6f}")
print(f"  X_val mean: {X_val_scaled.mean():.6f}, std: {X_val_scaled.std():.6f}")
print(f"  X_test mean: {X_test_scaled.mean():.6f}, std: {X_test_scaled.std():.6f}")
print(f"\n✓ All splits standardized using training data statistics")

Feature standardization complete:
  X_train mean: 0.000000, std: 1.000000
  X_val mean: 0.191136, std: 1.972306
  X_test mean: 0.179499, std: 1.465364

✓ All splits standardized using training data statistics


In [ ]:
# Define evaluation metrics function
def compute_metrics(y_true, y_proba, split_name=""):
    """
    Compute comprehensive metrics for binary classification.

    Args:
        y_true: Binary labels (0 or 1)
        y_proba: Predicted probabilities for positive class (shape: (n,))
        split_name: Name of split (for printing)

    Returns:
        dict with metrics
    """
    # Ensure y_proba is 1D
    if len(y_proba.shape) > 1:
        y_proba = y_proba[:, 1]

    y_pred = (y_proba >= 0.5).astype(int)

    pr_auc = average_precision_score(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    brier = brier_score_loss(y_true, y_proba)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "brier": brier,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    }


print("✓ Evaluation metrics function defined")

✓ Evaluation metrics function defined


In [ ]:
# Unified training and evaluation framework
def train_and_evaluate(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    class_weight_dict,
    model_name,
    verbose=True,
):
    """
    Train a model and evaluate on all splits.

    Args:
        model: Scikit-learn model (unfitted)
        X_train, y_train: Training data
        X_val, y_val: Validation data
        X_test, y_test: Test data
        class_weight_dict: Class weights dict {0: w0, 1: w1}
        model_name: Name of model for printing
        verbose: Print results

    Returns:
        dict with model and metrics for all splits
    """
    # Update model with class weights if supported
    if hasattr(model, "class_weight"):
        model.class_weight = class_weight_dict

    # Train
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)

    # Predict on all splits
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_val_proba = model.predict_proba(X_val)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]

    # Compute metrics
    train_metrics = compute_metrics(y_train, y_train_proba, "train")
    val_metrics = compute_metrics(y_val, y_val_proba, "val")
    test_metrics = compute_metrics(y_test, y_test_proba, "test")

    if verbose:
        print(f"\n{model_name} Results:")
        print(
            f"  Train - PR-AUC: {train_metrics['pr_auc']:.4f}, ROC-AUC: {train_metrics['roc_auc']:.4f}, Brier: {train_metrics['brier']:.4f}"
        )
        print(
            f"  Val   - PR-AUC: {val_metrics['pr_auc']:.4f}, ROC-AUC: {val_metrics['roc_auc']:.4f}, Brier: {val_metrics['brier']:.4f}"
        )
        print(
            f"  Test  - PR-AUC: {test_metrics['pr_auc']:.4f}, ROC-AUC: {test_metrics['roc_auc']:.4f}, Brier: {test_metrics['brier']:.4f}"
        )

    return {
        "model_name": model_name,
        "model": model,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "y_train_proba": y_train_proba,
        "y_val_proba": y_val_proba,
        "y_test_proba": y_test_proba,
    }


print("✓ Training framework defined")

✓ Training framework defined


In [ ]:
# MODEL 1: Logistic Regression (L2, Class-Weighted)
print("=" * 70)
print("MODEL 1: Logistic Regression (L2, Class-Weighted)")
print("=" * 70)

lr_model = LogisticRegression(
    penalty="l2",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    solver="lbfgs",
    verbose=0,
)

results_lr = train_and_evaluate(
    lr_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    X_test_scaled,
    y_test,
    class_weight_dict,
    "Logistic Regression (L2)",
    verbose=True,
)

MODEL 1: Logistic Regression (L2, Class-Weighted)

Training Logistic Regression (L2)...

Logistic Regression (L2) Results:
  Train - PR-AUC: 0.8433, ROC-AUC: 0.9797, Brier: 0.0645
  Val   - PR-AUC: 0.3739, ROC-AUC: 0.8975, Brier: 0.1857
  Test  - PR-AUC: 0.1993, ROC-AUC: 0.8554, Brier: 0.1996


In [ ]:
# MODEL 2: Linear SGD (Logistic Loss, Class-Weighted)
print("\n" + "=" * 70)
print("MODEL 2: Linear SGD (Logistic Loss, Class-Weighted)")
print("=" * 70)

sgd_model = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    tol=1e-3,
    verbose=0,
    n_jobs=-1,
)

results_sgd = train_and_evaluate(
    sgd_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    X_test_scaled,
    y_test,
    class_weight_dict,
    "SGD (logistic loss)",
    verbose=True,
)


MODEL 2: Linear SGD (Logistic Loss, Class-Weighted)

Training SGD (logistic loss)...

SGD (logistic loss) Results:
  Train - PR-AUC: 0.8236, ROC-AUC: 0.9778, Brier: 0.0656
  Val   - PR-AUC: 0.3016, ROC-AUC: 0.8780, Brier: 0.1956
  Test  - PR-AUC: 0.1542, ROC-AUC: 0.8449, Brier: 0.1926


In [ ]:
# MODEL 3: Random Forest (class-weighted)
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)

results_rf = train_and_evaluate(
    rf_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    X_test_scaled,
    y_test,
    class_weight_dict,
    "Random Forest",
    verbose=True,
)


Training Random Forest...

Random Forest Results:
  Train - PR-AUC: 1.0000, ROC-AUC: 1.0000, Brier: 0.0016
  Val   - PR-AUC: 0.9294, ROC-AUC: 0.9778, Brier: 0.0171
  Test  - PR-AUC: 0.5440, ROC-AUC: 0.8478, Brier: 0.0297


In [ ]:
# MODEL 4: Gradient-Boosted Trees (XGBoost/LightGBM/fallback)
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

try:
    from xgboost import XGBClassifier

    gb_model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=pos_weight,
        random_state=42,
        n_jobs=-1,
    )
    gb_name = "XGBoost"
except Exception:
    try:
        from lightgbm import LGBMClassifier

        gb_model = LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        gb_name = "LightGBM"
    except Exception:
        gb_model = HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=500, max_depth=6, random_state=42
        )
        gb_name = "HistGradientBoosting (fallback)"

print(f"Using boosted model: {gb_name}")
results_gb = train_and_evaluate(
    gb_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    X_test_scaled,
    y_test,
    class_weight_dict,
    gb_name,
    verbose=True,
)

Using boosted model: HistGradientBoosting (fallback)

Training HistGradientBoosting (fallback)...

HistGradientBoosting (fallback) Results:
  Train - PR-AUC: 0.9997, ROC-AUC: 1.0000, Brier: 0.0019
  Val   - PR-AUC: 0.9334, ROC-AUC: 0.9787, Brier: 0.0179
  Test  - PR-AUC: 0.5543, ROC-AUC: 0.8514, Brier: 0.0357


In [ ]:
# MODEL 5: Feature-only MLP
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    alpha=1e-4,
    learning_rate_init=1e-3,
    batch_size=512,
    max_iter=80,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=8,
    random_state=42,
)

results_mlp = train_and_evaluate(
    mlp_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    X_test_scaled,
    y_test,
    class_weight_dict,
    "Feature-only MLP",
    verbose=True,
)


Training Feature-only MLP...

Feature-only MLP Results:
  Train - PR-AUC: 0.9881, ROC-AUC: 0.9974, Brier: 0.0082
  Val   - PR-AUC: 0.6241, ROC-AUC: 0.9193, Brier: 0.0506
  Test  - PR-AUC: 0.3863, ROC-AUC: 0.8320, Brier: 0.0354


In [ ]:
# RESULTS COMPARISON (all 5 models)
print("\n" + "=" * 70)
print("RESULTS COMPARISON: 5 BASELINE MODELS")
print("=" * 70)


results_all = [results_lr, results_sgd, results_rf, results_gb, results_mlp]


df_summary = pd.DataFrame(
    [
        {
            "Model": r["model_name"],
            "Train PR-AUC": r["train_metrics"]["pr_auc"],
            "Val PR-AUC": r["val_metrics"]["pr_auc"],
            "Test PR-AUC": r["test_metrics"]["pr_auc"],
            "Train Brier": r["train_metrics"]["brier"],
            "Val Brier": r["val_metrics"]["brier"],
            "Test Brier": r["test_metrics"]["brier"],
        }
        for r in results_all
    ]
).sort_values("Val PR-AUC", ascending=False)

print("\nRanked by Validation PR-AUC (primary):")
print(
    df_summary[
        ["Model", "Val PR-AUC", "Test PR-AUC", "Val Brier", "Test Brier"]
    ].to_string(index=False)
)

best_model_idx = df_summary["Val PR-AUC"].idxmax()
best_model_name = df_summary.loc[best_model_idx, "Model"]
best_val_pr_auc = df_summary.loc[best_model_idx, "Val PR-AUC"]

print("\n" + "=" * 70)
print(f"WINNER (highest Val PR-AUC): {best_model_name}")
print(f"Val PR-AUC: {best_val_pr_auc:.4f}")
print(f"Test PR-AUC: {df_summary.loc[best_model_idx, 'Test PR-AUC']:.4f}")
print("=" * 70)


RESULTS COMPARISON: 5 BASELINE MODELS

Ranked by Validation PR-AUC (primary):
                          Model  Val PR-AUC  Test PR-AUC  Val Brier  Test Brier
HistGradientBoosting (fallback)    0.933411     0.554346   0.017887    0.035749
                  Random Forest    0.929439     0.543988   0.017067    0.029740
               Feature-only MLP    0.624086     0.386287   0.050628    0.035393
       Logistic Regression (L2)    0.373944     0.199295   0.185729    0.199584
            SGD (logistic loss)    0.301557     0.154202   0.195598    0.192593

WINNER (highest Val PR-AUC): HistGradientBoosting (fallback)
Val PR-AUC: 0.9334
Test PR-AUC: 0.5543


In [ ]:
# SUMMARY (concise)
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)

best_model = results_all[best_model_idx]

print(f"Winner: {best_model['model_name']}")
print(
    f"Val PR-AUC: {best_model['val_metrics']['pr_auc']:.4f} | Test PR-AUC: {best_model['test_metrics']['pr_auc']:.4f}"
)
print(
    f"Val Brier: {best_model['val_metrics']['brier']:.4f} | Test Brier: {best_model['test_metrics']['brier']:.4f}"
)

print(
    "\nNext steps: add Models 3-5, then consider post-hoc calibration if Brier stays high."
)


SUMMARY
Winner: HistGradientBoosting (fallback)
Val PR-AUC: 0.9334 | Test PR-AUC: 0.5543
Val Brier: 0.0179 | Test Brier: 0.0357

Next steps: add Models 3-5, then consider post-hoc calibration if Brier stays high.
